# Chicago Taxi Data Analysis

## Introduction

In this project, we analyze taxi ride data from Chicago to identify patterns in taxi companies, trip destinations, and ride duration under different weather conditions.

The project is divided into two main parts:

1. Exploratory Data Analysis (EDA)
2. Statistical Hypothesis Testing

## Exploratory Data Analysis

### Load Data 

In [98]:
import pandas as pd
from pathlib import Path
import plotly.express as px
from scipy import stats as st

#### Company trips

This dataset contains information about taxi companies operating in Chicago and the total number of trips completed by each company on November 15 and 16, 2017. It will be used to identify the most active taxi companies and analyze how ride demand is distributed across providers.

In [ ]:
if Path('/datasets/project_sql_result_01.csv').exists():
    dataset_path = Path('/datasets')    # Platform Tripleten
else:
    dataset_path = Path('./datasets')    # Local development

company_trips = pd.read_csv(dataset_path / 'project_sql_result_01.csv')
company_trips.head()

,company_name,trips_amount
0,Flash Cab,19558
1,Taxi Affiliation Services,11422
2,Medallion Leasin,10367
3,Yellow Cab,9888
4,Taxi Affiliation Service Yellow,9299


In [5]:
company_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   company_name  64 non-null     str  
 1   trips_amount  64 non-null     int64
dtypes: int64(1), str(1)
memory usage: 1.1 KB


The data types is correct and there are not null values.

#### Neighborhood Trips

This dataset contains information about Chicago neighborhoods and the average number of taxi trips that ended in each neighborhood during November 2017. The data will be used to identify the most popular drop-off destinations and understand geographic patterns in taxi demand.

In [6]:
neighborhood_trips = pd.read_csv(dataset_path / 'project_sql_result_04.csv')

neighborhood_trips.head() 

,dropoff_location_name,average_trips
0,Loop,10727.466667
1,River North,9523.666667
2,Streeterville,6664.666667
3,West Loop,5163.666667
4,O'Hare,2546.900000


In [7]:
neighborhood_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dropoff_location_name  94 non-null     str    
 1   average_trips          94 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.6 KB


The data types is correct and there are not null values.

### Top 10 Neighborhoods

Identify the top 10 neighborhoods by number of trip drop-offs.

In [80]:
# Identify the top 10 neighborhoods with the highest number of trips
top_10_neighborhoods = neighborhood_trips.sort_values('average_trips', ascending=False).head(10)

top_10_neighborhoods

,dropoff_location_name,average_trips
0,Loop,10727.466667
1,River North,9523.666667
2,Streeterville,6664.666667
3,West Loop,5163.666667
4,O'Hare,2546.900000
5,Lake View,2420.966667
6,Grant Park,2068.533333
7,Museum Campus,1510.000000
8,Gold Coast,1364.233333
9,Sheffield & DePaul,1259.766667


In [ ]:
# plot the top 10 neighborhoods with the highest average number of trips
fig = px.bar(
    top_10_neighborhoods, 
    y='dropoff_location_name', 
    x='average_trips', 
    title='Top 10 Chicago Neighborhoods by Average Taxi Drop-offs',
    labels={'dropoff_location_name':'Neighborhood', 'average_trips':'Average Number of Trips'},
    orientation='h',
    #color_discrete_sequence=['#1f77b4'],
    color_discrete_sequence=['#0068c9'],
    #color_discrete_sequence=['#0078D4'],
    text='average_trips',
    template='plotly_white',
    #color='average_trips',
    #color_continuous_scale='Blues'
    )

# Customize the layout of the plot
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_tickformat=',',
    width=1000,
    height=600,
    margin=dict(l=50, r=20, t=60, b=40),
    coloraxis_showscale=False
)

# Add text labels to the bars
fig.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside'
)

fig.show()

#### Findings

- **Loop** is the most popular drop-off destination, averaging more than 10,000 trips.
- **River North** and **Streeterville** rank second and third, indicating high transportation demand in central business and tourist areas.
- A significant drop is observed after the top **four neighborhoods**, suggesting that taxi demand is concentrated in a relatively **small number** of destinations.

### Taxi Companies

Identify the taxi companies with the highest number of trips on November 15 and 16, 2017.

In [91]:
# plot the top taxi companies with highest number of trips
fig = px.bar(
    company_trips.head(10), 
    y='company_name', 
    x='trips_amount', 
    title='Top 10 Taxi Companies by Number of Trips',
    labels={'company_name':'Company', 'trips_amount':'Number of Trips'},
    orientation='h',
    #color_discrete_sequence=['#1f77b4'],
    color_discrete_sequence=['#0068c9'],
    #color_discrete_sequence=['#074985'],
    #color_discrete_sequence=['#0078D4'],
    text='trips_amount',
    template='plotly_white',
    #color='trips_amount',
    #color_continuous_scale='Blues'
    )

# Customize the layout of the plot
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_tickformat=',',
    width=1000,
    height=600,
    margin=dict(l=50, r=20, t=60, b=40),
    coloraxis_showscale=False
)

# Add text labels to the bars
fig.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside'
)

fig.show()

#### Findings

- **Flash Cab** was the leading taxi company with 19,558 trips, significantly outperforming all competitors.
- Flash Cab completed approximately **71%** more trips than the second-place company.
- Companies ranked from second to seventh show relatively similar trip volumes, ranging from about **8,400 to 11,400** trips.
- The market appears to be concentrated, with **one dominant company** and several competitors showing similar trip volumes.

## Hypothesis Testing

Determine whether the average duration of rides from the Loop to O'Hare Airport changes on rainy Saturdays.

### Load Data

#### Loop to O'Hare Trips

This dataset contains information about taxi rides from the Loop neighborhood to O'Hare International Airport. For each trip, it includes the pickup timestamp, weather conditions at the start of the ride, and trip duration in seconds. The dataset will be used to test whether rainy Saturdays affect average travel times.

In [8]:
weather_trips = pd.read_csv(dataset_path / 'project_sql_result_07.csv')

weather_trips.head()

,start_ts,weather_conditions,duration_seconds
0,2017-11-25 16:00:00,Good,2410.0
1,2017-11-25 14:00:00,Good,1920.0
2,2017-11-25 12:00:00,Good,1543.0
3,2017-11-04 10:00:00,Good,2512.0
4,2017-11-11 07:00:00,Good,1440.0


In [9]:
weather_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_ts            1068 non-null   str    
 1   weather_conditions  1068 non-null   str    
 2   duration_seconds    1068 non-null   float64
dtypes: float64(1), str(2)
memory usage: 25.2 KB


The **start_ts** column is stored as a string (`object`) rather than a datetime data type.

In [12]:
# Convert to datetime
weather_trips['start_ts'] = pd.to_datetime(weather_trips['start_ts']) 

weather_trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   start_ts            1068 non-null   datetime64[us]
 1   weather_conditions  1068 non-null   str           
 2   duration_seconds    1068 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 25.2 KB


### Define Hypothesis

**Null Hypothesis (H₀)**

>   The average duration of rides from the Loop to O'Hare is the **same** on rainy and non-rainy Saturdays.

**Alternative Hypothesis (H₁)**

>   The average duration of rides from the Loop to O'Hare is **different** on rainy and non-rainy Saturdays.

**Method** 

* The analysis will be conducted using a **Student's t-test** to compare the mean trip duration between rainy and non-rainy Saturdays. Prior to the t-test, **Levene's** test will be applied to verify that the variances of both groups are statistically similar, satisfying the assumption of homogeneity of variances.
* A **one-tailed test** will be used because the alternative hypothesis states that the average trip duration on rainy Saturdays is different than the average trip duration on non-rainy Saturdays.

**Significance Level** 

* alpha = `0.05` 
* A significance level of 5% is the most commonly accepted threshold for statistical testing.

In [ ]:
# Group by weather conditions and calculate mean, count, and sum of trip durations
weather_trips.groupby('weather_conditions').agg({'duration_seconds': ['mean','count','sum']})

duration_seconds                 
                               mean count        sum
weather_conditions                                  
Bad                     2427.205556   180   436897.0
Good                    1999.675676   888  1775712.0

In [103]:
# Separate the data into two groups: trips with rain and trips without rain
rain_trips = weather_trips[weather_trips['weather_conditions'] == 'Bad']
no_rain_trips = weather_trips[weather_trips['weather_conditions'] == 'Good']

# Significance level
alpha = 0.05

# Perform a Levene's test to check for equal variances
levene_test = st.levene(rain_trips['duration_seconds'], no_rain_trips['duration_seconds'])
equal_var = levene_test.pvalue > alpha

if equal_var:
    print("Equal variances assumption is met.")
else:
    print("Equal variances assumption is not met.")
print(f"P-value from Levene's test: {levene_test.pvalue:.2f}")

Equal variances assumption is met.
P-value from Levene's test: 0.53


In [106]:
# Perform a t-test 
t_test = st.ttest_ind(rain_trips['duration_seconds'], no_rain_trips['duration_seconds'], equal_var=equal_var)
print(f"P-value from t-test: {t_test.pvalue:.2f}")

if t_test.pvalue < alpha:
    print("Reject the null hypothesis: There is a significant difference in trip durations between rainy and non-rainy conditions.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in trip durations between rainy and non-rainy conditions.")

P-value from t-test: 0.00
Reject the null hypothesis: There is a significant difference in trip durations between rainy and non-rainy conditions.


In [ ]:
# Histogram of trip durations for rainy and non-rainy conditions
fig = px.histogram(
    weather_trips,
    x='duration_seconds',
    color='weather_conditions',
    histnorm='probability density',
    barmode='overlay',
    opacity=0.85,
    title='Distribution of Trip Durations by Weather Conditions'
    
    
)

fig.show()